# T2 — Decisiones de criterio (*judgment calls*) y análisis de estabilidad

**Grupo 01** — Aplicaciones de IA en Estructuras, UPC

El marco PCS (Yu y Barter, 2024) exige que las conclusiones se mantengan ante
**perturbaciones razonables** de las decisiones de análisis. En el T1 (sección 7.3)
identificamos cinco decisiones de criterio. Este notebook **cuantifica el efecto de cada
una** antes de fijarla, para que la elección quede justificada con evidencia y no solo
por costumbre.

**Herramienta de medición:** una regresión lineal de referencia con la forma simplificada
de GMPE del informe T1 (sección 5.3):

$$\ln(PGA) = c_0 + c_1 M_w + c_2 \ln\sqrt{R^2 + h^2} + c_3 \ln(V_{S30})$$

No es el modelo final: se usa como **instrumento** para ver cuánto cambian los
coeficientes físicos cuando cambia una decisión. Si un coeficiente cambia mucho, esa
decisión es sensible y debe justificarse con especial cuidado.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression

SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")

## 1. Carga y limpieza base

Misma limpieza del notebook `01_inventario_eda`.

In [ ]:
RUTA = "../data/Updated_NGA_West2_Flatfile_RotD50_d005_public_version.xlsx"

COLUMNAS = {
    "EQID": "EQID",
    "Earthquake Magnitude": "Mw",
    "ClstD (km)": "Rrup",
    "Joyner-Boore Dist. (km)": "Rjb",
    "Vs30 (m/s) selected for analysis": "Vs30",
    "Measured/Inferred Class": "vs30_clase",
    "Lowest Usable Freq - Ave. Component (Hz)": "f_min",
    "Spectra Quality Flag": "calidad",
    "Mechanism Based on Rake Angle": "mecanismo",
    "Hypocenter Depth (km)": "prof_hipo",
    "CRjb": "CRjb", "CRjb.1": "CRjb_1", "CRjb.2": "CRjb_2",
    "CRjb.3": "CRjb_3", "CRjb.4": "CRjb_4", "CRjb.5": "CRjb_5",
    "PGA (g)": "PGA",
}

df = pd.read_excel(RUTA, usecols=list(COLUMNAS)).rename(columns=COLUMNAS)

numericas = ["EQID", "Mw", "Rrup", "Rjb", "Vs30", "f_min", "calidad",
             "mecanismo", "prof_hipo", "PGA"]
df[numericas] = df[numericas].replace(-999, np.nan)

base = df.dropna(subset=["PGA", "Mw", "Rrup", "Rjb", "Vs30",
                         "mecanismo", "prof_hipo", "EQID"]).copy()
base["lnPGA"] = np.log(base["PGA"])

print(f"Dataset base: {len(base):,} registros, {base['EQID'].nunique()} sismos")

## 2. Instrumento de medición

Función que ajusta la forma GMPE simplificada y devuelve los coeficientes y la
desviación estándar de los residuos. Se usa $h = 6$ km como término de saturación,
un valor ilustrativo que deberán contrastar con la literatura (por ejemplo, el valor
calibrado en Boore et al., 2014).

In [ ]:
H = 6.0  # km, término de saturación (ver nota anterior)

def ajustar(datos, distancia="Rrup", nombre=""):
    """Ajusta ln(PGA) = c0 + c1*Mw + c2*ln(sqrt(R^2+h^2)) + c3*ln(Vs30)."""
    R = np.sqrt(datos[distancia] ** 2 + H ** 2)
    X = np.column_stack([datos["Mw"], np.log(R), np.log(datos["Vs30"])])
    modelo = LinearRegression().fit(X, datos["lnPGA"])
    residuos = datos["lnPGA"] - modelo.predict(X)
    return {
        "escenario": nombre,
        "registros": len(datos),
        "sismos": datos["EQID"].nunique(),
        "c1_Mw": modelo.coef_[0],
        "c2_dist": modelo.coef_[1],
        "c3_Vs30": modelo.coef_[2],
        "sigma": residuos.std(),
    }

ref = ajustar(base, nombre="base")
pd.DataFrame([ref]).round(3)

**Verificación de signos físicos.** Antes de seguir, los coeficientes deben tener el
signo esperado: $c_1 > 0$ (más magnitud, más PGA), $c_2 < 0$ (más distancia, menos PGA)
y $c_3 < 0$ (suelo más rígido, menos amplificación). Si alguno sale con el signo
contrario, hay un error en los datos.

In [ ]:
assert ref["c1_Mw"] > 0,   "c1 debería ser positivo"
assert ref["c2_dist"] < 0, "c2 debería ser negativo"
assert ref["c3_Vs30"] < 0, "c3 debería ser negativo"
print("Signos físicos correctos.")

## 3. Decisión 1 — Rango de magnitud y distancia

Los sismos pequeños y los registros muy lejanos son abundantes pero poco relevantes para
diseño, y pueden dominar el ajuste. Evaluamos varios recortes razonables.

In [ ]:
escenarios_md = {
    "Sin filtro":            base,
    "Mw >= 3, R <= 400 km":  base[(base.Mw >= 3) & (base.Rrup <= 400)],
    "Mw >= 4":               base[base.Mw >= 4],
    "Mw >= 4, R <= 300 km":  base[(base.Mw >= 4) & (base.Rrup <= 300)],
    "Mw >= 4, R <= 200 km":  base[(base.Mw >= 4) & (base.Rrup <= 200)],
}

tabla_md = pd.DataFrame([ajustar(d, nombre=n) for n, d in escenarios_md.items()])
tabla_md.round(3)

**Para discutir en el grupo:** ¿cuánto cambia cada coeficiente entre escenarios?
Un rango muy amplio conserva datos pero incluye registros fuera del interés de diseño;
uno muy estrecho reduce la muestra de sismos grandes, que ya es escasa
(ver notebook 01, sección 9).

## 4. Decisión 2 — Rrup o Rjb

`Rrup` es la distancia más corta a la superficie de ruptura; `Rjb` es la distancia
horizontal a su proyección en superficie. Difieren sobre todo para sitios cercanos a
fallas inclinadas o profundas.

In [ ]:
sub = base[(base.Mw >= 4) & (base.Rrup <= 200)]

tabla_dist = pd.DataFrame([
    ajustar(sub, "Rrup", "Rrup"),
    ajustar(sub, "Rjb",  "Rjb"),
])
tabla_dist.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(sub["Rrup"], sub["Rjb"], s=3, alpha=0.3)
lim = [0.1, 250]
ax.plot(lim, lim, "r--", lw=1, label="Rjb = Rrup")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Rrup (km)"); ax.set_ylabel("Rjb (km)")
ax.set_title("Diferencia entre las medidas de distancia")
ax.legend()
plt.tight_layout()
plt.savefig("../results/rrup_vs_rjb.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Decisión 3 — Vs30 medido o inferido

La columna `Measured/Inferred Class` contiene códigos que describen cómo se obtuvo el
Vs30. **Antes de usar esta decisión, el grupo debe confirmar el significado de cada código
en `NGA_West2_SiteDatabaseDocumentation_V032.xlsx`**, descargado junto con los datos de
apoyo del PEER. Aquí se evalúa a modo exploratorio el subconjunto de clase `0`.

In [ ]:
sub = base[(base.Mw >= 4) & (base.Rrup <= 200)]
clase0 = sub[sub["vs30_clase"].astype(str) == "0"]

tabla_vs = pd.DataFrame([
    ajustar(sub,    nombre="Todas las clases"),
    ajustar(clase0, nombre="Solo clase 0 (confirmar significado)"),
    ajustar(sub[(sub.Vs30 >= 150) & (sub.Vs30 <= 1500)], nombre="Vs30 entre 150 y 1500 m/s"),
])
tabla_vs.round(3)

**Observación:** filtrar por clase reduce la muestra a una fracción de los sismos.
Es la decisión con mayor impacto en el tamaño del dataset y debe pesarse contra la
ganancia en confiabilidad del Vs30.

## 6. Decisión 4 — Réplicas

El flatfile **no incluye una columna directa** que indique si un sismo es principal o
réplica. Sí contiene las columnas `CRjb` a `CRjb.5`, relacionadas con la clasificación
de eventos de NGA-West2, cuyo significado exacto no está documentado en el archivo.

**Pendiente del grupo:** consultar el reporte PEER 2013/03 (Ancheta et al.) y la
literatura sobre clasificación de réplicas en NGA-West2 para interpretar estas columnas
antes de aplicar cualquier filtro. No se debe filtrar con un criterio no verificado.

In [ ]:
cols_crjb = ["CRjb", "CRjb_1", "CRjb_2", "CRjb_3", "CRjb_4", "CRjb_5"]
base[cols_crjb].describe().T.round(2)

## 7. Decisión 5 — Calidad del registro

Para la PGA, la frecuencia mínima utilizable (`f_min`) es menos crítica que para
periodos largos, porque la PGA está dominada por altas frecuencias. Revisamos además el
indicador de calidad espectral.

In [ ]:
print("Indicador de calidad espectral:")
print(base["calidad"].value_counts(dropna=False))

sub = base[(base.Mw >= 4) & (base.Rrup <= 200)]
tabla_cal = pd.DataFrame([
    ajustar(sub, nombre="Todos"),
    ajustar(sub[sub["calidad"] == 0], nombre="Solo calidad = 0"),
])
tabla_cal.round(3)

## 8. Resumen de estabilidad

Todas las perturbaciones en una sola figura. Esta es la evidencia central del principio
de **Estabilidad** del marco PCS: muestra qué tanto dependen los coeficientes físicos de
cada decisión.

In [ ]:
resumen = pd.concat([tabla_md, tabla_dist, tabla_vs, tabla_cal], ignore_index=True)

fig, ax = plt.subplots(1, 3, figsize=(16, 6), sharey=True)
for eje, col, titulo in zip(ax,
        ["c1_Mw", "c2_dist", "c3_Vs30"],
        ["c1 (magnitud)", "c2 (distancia)", "c3 (Vs30)"]):
    eje.barh(resumen["escenario"], resumen[col], color="steelblue")
    eje.axvline(ref[col], color="red", ls="--", lw=1, label="base")
    eje.set_title(titulo)
    eje.legend()
plt.suptitle("Sensibilidad de los coeficientes físicos a las decisiones de criterio")
plt.tight_layout()
plt.savefig("../results/estabilidad_judgment_calls.png", dpi=150, bbox_inches="tight")
plt.show()

print("Rango de variación de cada coeficiente entre escenarios:")
for col in ["c1_Mw", "c2_dist", "c3_Vs30", "sigma"]:
    print(f"  {col:8s}: {resumen[col].min():.3f} a {resumen[col].max():.3f}")

## 9. Lectura de los resultados

Preguntas que el grupo debe responder por escrito para el informe T2:

1. **¿Qué coeficiente es el más sensible?** Comparar el rango de variación de cada uno.
2. **¿Mantienen todos los escenarios los signos físicos correctos?** Si sí, la relación
   física es robusta aunque su magnitud varíe.
3. **¿Por qué la desviación estándar de los residuos es mayor que la de las GMPE
   publicadas?** Pista: la forma simplificada no incluye saturación por magnitud, efectos
   no lineales del suelo ni términos del mecanismo de falla. Esto **justifica el uso de
   Machine Learning**, que puede capturar esos efectos sin especificarlos a priori.

## 10. Registro de decisiones

Completar esta tabla en grupo. Cada fila debe tener una justificación técnica, idealmente
con referencia bibliográfica. Esta tabla pasa directamente al informe T2.

| # | Decisión | Opciones evaluadas | Elección | Justificación | Referencia |
|---|---|---|---|---|---|
| 1 | Rango Mw y distancia | ver sección 3 | | | |
| 2 | Medida de distancia | Rrup / Rjb | | | |
| 3 | Vs30 inferido | todas / solo medidas | | | |
| 4 | Réplicas | pendiente de interpretar CRjb | | | |
| 5 | Calidad del registro | todos / calidad 0 | | | |

> Las decisiones descartadas **no se eliminan**: se conservan como escenarios
> alternativos para el análisis de estabilidad del T3, donde se verificará si la
> comparación entre modelos de ML cambia al modificarlas.